# Choice questions: LoRA fine-tuning + a scoring head

**What.** A model that answers **grounded choice questions** about a short text: yes / no, one of N options, or a level on an ordered scale. Every answer is in the text; no world knowledge is needed. This is the job the Strands decider does, and the E12 test set of `modernbert.ipynb` measures it.

**Why a new notebook.** In `modernbert.ipynb` the encoder is **frozen** and only a small head is trained on cached vectors. For choice questions that approach plateaued: run 16 scored 113 / 170 and run 17 scored 114 / 170, even with six times more training data, against the decider's 160. The questions it misses need *reading* ("Which main did the customer order?", "Who scored highest?"), and a frozen encoder's averaged vectors do not carry those answers. Here the **encoder learns too**, through LoRA adapters.

**Scope.** Choice questions only. No chat history, no agent routing and no detector.

**How it works.** One encoder pass per option, as a cross-encoder:

```text
"Who scored highest?"   text: "Ana scored 72, Ben scored 85 and Cara scored 79."
[text] [question] [option: Ana]  -> encoder (+ LoRA) -> mean of token vectors -> head -> score 0.3
[text] [question] [option: Ben]  -> encoder (+ LoRA) -> mean of token vectors -> head -> score 2.1
[text] [question] [option: Cara] -> encoder (+ LoRA) -> mean of token vectors -> head -> score 1.4
                                                         softmax over the options -> Ben
```

The loss is the cross-entropy of that softmax (listwise, over 2–5 options). Its gradient trains the head **and** the LoRA adapters inside the encoder; the original encoder weights stay frozen.

**Data.**
- Training: `data/modernbert_router/choice_train/items.json`, 9,113 questions from 24 families (yes / no balanced 50 / 50). 10% of the **texts** are held back as a dev set for early stopping; a text is never on both sides.
- Test: E12, `choice_test/items.json`, 170 questions, never used for training or for any choice made here.

**Compared with** (the exact same 170 E12 questions): run 14 frozen router, zero-shot, 96; run 17 frozen encoder + trained choice head, 114; Strands decider, 160.

## 1 — Setup

Settings are environment variables, so a run can be changed without editing code:

| Setting | Default | Meaning |
|---|---|---|
| `CHOICE_ENCODER` | `modernbert-large` | the encoder to fine-tune: `modernbert-large`, `modernbert-base` or `nemotron-1b` |
| `LORA_R`, `LORA_ALPHA` | 16, 32 | LoRA rank and scale (scale = alpha / r) |
| `EPOCHS` | 3 | passes over the training questions |
| `LR`, `HEAD_LR` | 2e-4, 1e-3 | learning rates of the LoRA adapters and of the head |
| `BATCH_Q` | 16 | questions per step (each brings 2–5 option passes) |
| `MAX_LEN` | 160 | tokens per (text, question, option) input |
| `CHOICE_SMOKE` | 0 | 1 = a tiny code check on a CPU, not a real run |

In [1]:
!pip install -q "transformers==5.17.0"
import json, math, os, random, subprocess, time
from collections import Counter
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

SMOKE = os.environ.get("CHOICE_SMOKE") == "1"
ENCODER = os.environ.get("CHOICE_ENCODER", "modernbert-base" if SMOKE else "modernbert-large")
LORA_R, LORA_ALPHA, LORA_DROPOUT = int(os.environ.get("LORA_R", "16")), float(os.environ.get("LORA_ALPHA", "32")), 0.1
EPOCHS = int(os.environ.get("EPOCHS", "1" if SMOKE else "3"))
LR, HEAD_LR = float(os.environ.get("LR", "2e-4")), float(os.environ.get("HEAD_LR", "1e-3"))
BATCH_Q = int(os.environ.get("BATCH_Q", "4" if SMOKE else "16"))
MAX_LEN = int(os.environ.get("MAX_LEN", "160"))
SEED = int(os.environ.get("SEED", "42"))
K_MAX = 5

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
random.seed(SEED); torch.manual_seed(SEED)
RUN_LOG = Path("choice_lora_run.log")


def log(message):
    line = f"[{time.strftime('%H:%M:%S')}] {message}"
    print(line, flush=True)
    with open(RUN_LOG, "a") as fh:
        fh.write(line + "\n")


def gpu():
    return f"GPU {torch.cuda.memory_allocated(DEVICE) / 1e9:.1f} GB (peak {torch.cuda.max_memory_allocated(DEVICE) / 1e9:.1f})" if AMP else "CPU"


import transformers
log(f"torch {torch.__version__} | transformers {transformers.__version__} | device {DEVICE}"
    + (f" ({torch.cuda.get_device_name(DEVICE)})" if AMP else "") + f" | encoder {ENCODER}" + (" | SMOKE CHECK" if SMOKE else ""))

[08:11:35] torch 2.11.0+cu128 | transformers 5.17.0 | device cuda:0 (Tesla T4) | encoder modernbert-large


## 2 — Data

The data files live in the repo. On Kaggle they are cloned (or refreshed) from the experiment branch. The training questions are split **by text** into train and dev, so that the same text, asked different questions, never appears on both sides.

In [2]:
DATA_BRANCH = "exp/phase9-choice-lora"
DATA_DIR = Path("data/modernbert_router")
if not DATA_DIR.exists():
    repo = Path("llms-the-hard-way")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", DATA_BRANCH,
                        "https://github.com/skamalj/llms-the-hard-way.git", str(repo)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", DATA_BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo), "checkout", "-B", DATA_BRANCH, "FETCH_HEAD"], check=True)
    DATA_DIR = repo / "data" / "modernbert_router"

train_all = json.load(open(DATA_DIR / "choice_train" / "items.json"))["items"]
e12 = json.load(open(DATA_DIR / "choice_test" / "items.json"))["items"]

texts = sorted({it["state"] for it in train_all})
random.Random(SEED).shuffle(texts)
dev_texts = set(texts[: len(texts) // 10])
train_items = [it for it in train_all if it["state"] not in dev_texts]
dev_items = [it for it in train_all if it["state"] in dev_texts]
assert not {it["state"] for it in train_items} & {it["state"] for it in e12}, "no E12 text in training"
if SMOKE:
    train_items, dev_items, e12 = train_items[::300], dev_items[::100], e12[::20]

for name, items in [("train", train_items), ("dev", dev_items), ("E12 test", e12)]:
    print(f"{name:8s}: {len(items):5d} questions | " + ", ".join(f"{t} {n}" for t, n in Counter(it["type"] for it in items).items())
          + f" | options per question: {dict(sorted(Counter(len(it['options']) for it in items).items()))}")
print("\nexample:", {k: train_items[0][k] for k in ("state", "question", "options", "answer")})

train   :  8197 questions | noul 5000, choice 2564, score 633 | options per question: {2: 5728, 3: 1960, 4: 499, 5: 10}
dev     :   916 questions | noul 572, choice 268, score 76 | options per question: {2: 659, 3: 184, 4: 73}
E12 test:   170 questions | noul 58, choice 59, score 53 | options per question: {2: 67, 3: 79, 4: 14, 5: 10}

example: {'state': 'The staff were brilliant and the room was spotless.', 'question': 'Is the writer happy?', 'options': ['yes', 'no'], 'answer': 'yes'}


From https://github.com/skamalj/llms-the-hard-way
 * branch            exp/phase9-choice-lora -> FETCH_HEAD
Switched to a new branch 'exp/phase9-choice-lora'


## 3 — The encoder with LoRA adapters, and the scoring head

**LoRA, written out.** Every chosen linear layer `W` of the encoder (frozen) gets a trainable low-rank update:

```text
y = W x  +  (alpha / r) · B (A x)        A: [r, in], B: [out, r], r = 16
```

- `A` starts random and `B` starts at **zero**, so at step 0 the model is exactly the pretrained encoder.
- Only `A` and `B` (and the head) are trained: a few million parameters instead of hundreds of millions.
- **ModernBERT:** adapters on the attention projections `Wqkv` and `Wo` and the MLP layers `Wi` and `Wo` of all 28 layers. **Nemotron (Llama):** on `q/k/v/o_proj` and `gate/up/down_proj`.

**The head.** The mean of the encoder's output vectors over the real tokens → `Dropout → Linear(H, 256) → GELU → Dropout → Linear(256, 1)`: one score per (text, question, option).

In [3]:
from transformers import AutoModel, AutoTokenizer

ENCODERS = {
    "modernbert-large": {"name": "answerdotai/ModernBERT-large", "sep": " [SEP] ", "prefix": "", "dtype": torch.float32,
                         "targets": {"Wqkv", "Wo", "Wi"}, "remote": False},
    "modernbert-base":  {"name": "answerdotai/ModernBERT-base", "sep": " [SEP] ", "prefix": "", "dtype": torch.float32,
                         "targets": {"Wqkv", "Wo", "Wi"}, "remote": False},
    "nemotron-1b":      {"name": "nvidia/llama-nemotron-embed-1b-v2", "sep": "\n", "prefix": "query: ", "dtype": torch.float16,
                         "targets": {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}, "remote": True},
}
SPEC = ENCODERS[ENCODER]


class LoRALinear(nn.Module):
    """A frozen linear layer plus a trainable low-rank update: y = W x + (alpha / r) * B (A x)."""

    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.scale, self.drop = alpha / r, nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.t().to(x.dtype) @ self.B.t().to(x.dtype)) * self.scale


def add_lora(model, targets, r, alpha, dropout):
    """Replace every nn.Linear whose attribute name is in `targets` with a LoRALinear wrapper -> number replaced."""
    found = []
    for module in list(model.modules()):
        for child_name, child in list(module.named_children()):
            if child_name in targets and isinstance(child, nn.Linear):
                found.append((module, child_name, child))
    for module, child_name, child in found:
        setattr(module, child_name, LoRALinear(child, r, alpha, dropout))
    return len(found)


tokenizer = AutoTokenizer.from_pretrained(SPEC["name"], trust_remote_code=SPEC["remote"])
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
encoder = AutoModel.from_pretrained(SPEC["name"], trust_remote_code=SPEC["remote"], dtype=SPEC["dtype"]).to(DEVICE)
encoder.requires_grad_(False)                                              # the pretrained weights stay frozen
n_lora = add_lora(encoder, SPEC["targets"], LORA_R, LORA_ALPHA, LORA_DROPOUT)
encoder.to(DEVICE)
assert n_lora > 0, "no target layers found: check the layer names for this encoder"
H = encoder.config.hidden_size
head = nn.Sequential(nn.Dropout(0.1), nn.Linear(H, 256), nn.GELU(), nn.Dropout(0.1), nn.Linear(256, 1)).to(DEVICE)

lora_params = [p for n, p in encoder.named_parameters() if p.requires_grad]
n_frozen = sum(p.numel() for p in encoder.parameters() if not p.requires_grad)
log(f"{SPEC['name']}: {n_lora} linear layers with LoRA (r {LORA_R}, alpha {LORA_ALPHA}) | trainable: LoRA "
    f"{sum(p.numel() for p in lora_params) / 1e6:.2f} M + head {sum(p.numel() for p in head.parameters()) / 1e6:.2f} M | "
    f"frozen {n_frozen / 1e6:.0f} M | {gpu()}")

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-large
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[08:11:41] answerdotai/ModernBERT-large: 112 linear layers with LoRA (r 16, alpha 32.0) | trainable: LoRA 7.20 M + head 0.26 M | frozen 395 M | GPU 1.6 GB (peak 1.6)


## 4 — Input format and scoring

One input per option, in the same format as `modernbert.ipynb` (prefix and separator of the chosen encoder):

```text
user: <text> [SEP] question: <question> [SEP] candidate: <option>
```

`score_batch` turns a batch of questions into all their option inputs, runs the encoder **once** over all of them (one batched forward pass), and places each option's score on a `[questions, 5]` grid. Unused slots (a yes / no question has 2 options) get −10,000, so the softmax ignores them.

In [4]:
def build_input(item, option):
    sep = SPEC["sep"]
    return f"{SPEC['prefix']}user: {item['state']}{sep}question: {item['question']}{sep}candidate: {option}"


def score_batch(items):
    """-> scores [len(items), K_MAX] (padded slots at -1e4), from one batched encoder pass over every option."""
    inputs, owner, slot = [], [], []
    for b, it in enumerate(items):
        for k, opt in enumerate(it["options"]):
            inputs.append(build_input(it, opt)); owner.append(b); slot.append(k)
    enc = tokenizer(inputs, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
        hidden = encoder(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"]).last_hidden_state
    m = enc["attention_mask"].unsqueeze(-1).float()
    pooled = (hidden.float() * m).sum(1) / m.sum(1)
    s = head(pooled).squeeze(-1)
    scores = torch.full((len(items), K_MAX), -1e4, device=DEVICE)
    scores[torch.tensor(owner, device=DEVICE), torch.tensor(slot, device=DEVICE)] = s
    return scores


def targets(items):
    return torch.tensor([it["options"].index(it["answer"]) for it in items], device=DEVICE)


ex = train_items[0]
print(build_input(ex, ex["options"][0]))
print("tokens:", len(tokenizer(build_input(ex, ex["options"][0]))["input_ids"]),
      "| longest training input:", max(len(tokenizer(build_input(it, o))["input_ids"]) for it in train_items[:2000] for o in it["options"]))
with torch.no_grad():
    print("scores before training (random head):", score_batch(train_items[:2]).cpu())

user: The staff were brilliant and the room was spotless. [SEP] question: Is the writer happy? [SEP] candidate: yes
tokens: 29 | longest training input: 60
scores before training (random head): tensor([[ 3.3276e-02,  2.8562e-02, -1.0000e+04, -1.0000e+04, -1.0000e+04],
        [ 6.3209e-02,  1.4432e-01,  1.2743e-02, -1.0000e+04, -1.0000e+04]])


## 5 — Training

- **Loss:** listwise cross-entropy over each question's options.
- **Optimizer:** AdamW, learning rate `LR` for the LoRA adapters and `HEAD_LR` for the head, weight decay 0.01, 6% linear warm-up then linear decay, gradient clipping at 1.0. Mixed precision (fp16) on the GPU.
- **Early stopping:** after every epoch the dev questions (texts never seen in training) are scored, and the LoRA + head weights with the best dev accuracy are kept.

**Expect.** A progress line every 50 steps (loss, speed, time left, GPU memory) and dev accuracy per question type after each epoch.

In [5]:
import copy


def evaluate(items, batch=32):
    """-> (correct per item [list of bool], probabilities per item)."""
    encoder.eval(); head.eval()
    ok, probs = [], []
    with torch.no_grad():
        for s0 in range(0, len(items), batch):
            chunk = items[s0:s0 + batch]
            p = torch.softmax(score_batch(chunk), dim=1).cpu()
            for it, pi in zip(chunk, p):
                pi = pi[:len(it["options"])]
                probs.append(pi.tolist()); ok.append(it["options"][int(pi.argmax())] == it["answer"])
    return ok, probs


def by_type(items, ok):
    return " | ".join(f"{t} {sum(o for it, o in zip(items, ok) if it['type'] == t)}/{sum(it['type'] == t for it in items)}"
                      for t in ["noul", "choice", "score"])


trainable = lambda: [p for p in encoder.parameters() if p.requires_grad] + list(head.parameters())
opt = torch.optim.AdamW([{"params": [p for p in encoder.parameters() if p.requires_grad], "lr": LR},
                         {"params": head.parameters(), "lr": HEAD_LR}], weight_decay=0.01)
steps_per_epoch = math.ceil(len(train_items) / BATCH_Q)
total_steps = EPOCHS * steps_per_epoch
warmup = max(1, int(0.06 * total_steps))
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / warmup, max(0.0, (total_steps - s) / (total_steps - warmup))))
scaler = torch.amp.GradScaler("cuda", enabled=AMP)

ok0, _ = evaluate(dev_items)
log(f"dev before training: {sum(ok0)}/{len(ok0)} ({by_type(dev_items, ok0)}) | {total_steps} steps, {steps_per_epoch} per epoch")
best = (-1, None, 0)                                       # the untrained (random) head is never kept: the best TRAINED epoch is
t0, step = time.perf_counter(), 0
for epoch in range(1, EPOCHS + 1):
    encoder.train(); head.train()
    order = list(range(len(train_items)))
    random.Random(SEED + epoch).shuffle(order)
    running = []
    for s0 in range(0, len(order), BATCH_Q):
        batch = [train_items[i] for i in order[s0:s0 + BATCH_Q]]
        loss = F.cross_entropy(score_batch(batch), targets(batch))
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(trainable(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        running.append(loss.item()); step += 1
        if step % 50 == 0 or step == total_steps:
            el = time.perf_counter() - t0
            log(f"  epoch {epoch} step {step}/{total_steps} | loss {sum(running[-50:]) / len(running[-50:]):.4f} | "
                f"{step / el:.2f} steps/s | ~{(total_steps - step) / (step / el) / 60:.1f} min left | {gpu()}")
    ok, _ = evaluate(dev_items)
    log(f"epoch {epoch} done | train loss {sum(running) / len(running):.4f} | dev {sum(ok)}/{len(ok)} ({by_type(dev_items, ok)})")
    if sum(ok) > best[0]:
        best = (sum(ok), {"lora": {n: p.detach().clone() for n, p in encoder.named_parameters() if p.requires_grad},
                          "head": copy.deepcopy(head.state_dict())}, epoch)
train_time = time.perf_counter() - t0

if best[1] is not None:                                     # restore the best epoch's adapters and head
    params = dict(encoder.named_parameters())
    for n, v in best[1]["lora"].items():
        params[n].data.copy_(v)
    head.load_state_dict(best[1]["head"])
log(f"training done in {train_time / 60:.1f} min | best dev {best[0]}/{len(dev_items)} at epoch {best[2]}")

[08:11:51] dev before training: 399/916 (noul 276/572 | choice 93/268 | score 30/76) | 1539 steps, 513 per epoch
[08:12:08]   epoch 1 step 50/1539 | loss 0.8244 | 2.88 steps/s | ~8.6 min left | GPU 1.7 GB (peak 5.7)
[08:12:25]   epoch 1 step 100/1539 | loss 0.4364 | 2.88 steps/s | ~8.3 min left | GPU 1.7 GB (peak 5.8)
[08:12:42]   epoch 1 step 150/1539 | loss 0.2490 | 2.89 steps/s | ~8.0 min left | GPU 1.7 GB (peak 5.8)
[08:12:59]   epoch 1 step 200/1539 | loss 0.1070 | 2.91 steps/s | ~7.7 min left | GPU 1.7 GB (peak 5.8)
[08:13:16]   epoch 1 step 250/1539 | loss 0.1355 | 2.92 steps/s | ~7.4 min left | GPU 1.7 GB (peak 5.8)
[08:13:33]   epoch 1 step 300/1539 | loss 0.1187 | 2.92 steps/s | ~7.1 min left | GPU 1.7 GB (peak 5.8)
[08:13:50]   epoch 1 step 350/1539 | loss 0.0399 | 2.92 steps/s | ~6.8 min left | GPU 1.7 GB (peak 5.8)
[08:14:08]   epoch 1 step 400/1539 | loss 0.1001 | 2.92 steps/s | ~6.5 min left | GPU 1.7 GB (peak 5.8)
[08:14:25]   epoch 1 step 450/1539 | loss 0.0345 | 2.92 

## 6 — E12: the exact same 170 questions as `modernbert.ipynb` and the decider

E12 is scored once, with the weights chosen on the dev set. Each question is timed on its own (batch = one question, all of its options in one pass), the same way as the other models.

**Expect.** Correct per type, the score errors in levels, per category, the question groups (the same text asked different questions), every mistake, and a comparison table with run 14, run 17 and the decider.

In [6]:
encoder.eval(); head.eval()
score_batch([e12[0]])                                      # warm-up (not timed)
rows, t0 = [], time.perf_counter()
with torch.no_grad():
    for it in e12:
        p = torch.softmax(score_batch([it]), dim=1)[0, :len(it["options"])].cpu()
        top = int(p.argmax())
        row = {**it, "probs": p.tolist(), "pick": it["options"][top], "ok": it["options"][top] == it["answer"]}
        if it["type"] == "score":
            row["expected_level"] = float(sum(i * pi for i, pi in enumerate(p.tolist())))
            row["level_error"] = abs(top - it["answer_index"])
        rows.append(row)
if AMP:
    torch.cuda.synchronize(DEVICE)
ms = (time.perf_counter() - t0) * 1000 / len(e12)
log(f"E12: {len(rows)} questions, {ms:.0f} ms per question")

REFERENCE = {"run 14 frozen, zero-shot": {"noul": 29, "choice": 39, "score": 28},
             "run 17 frozen + choice head": {"noul": 38, "choice": 44, "score": 32},
             "Strands decider 2B": {"noul": 56, "choice": 56, "score": 48}}
counts = {t: sum(it["type"] == t for it in e12) for t in ["noul", "choice", "score"]}
ours = {t: sum(r["ok"] for r in rows if r["type"] == t) for t in counts}
print(f"\n{'E12':10s} | {'items':>5} | " + " | ".join(f"{k:>27s}" for k in REFERENCE) + f" | {'LoRA + head (this run)':>23s}")
for t in ["noul", "choice", "score"]:
    print(f"{t:10s} | {counts[t]:>5} | " + " | ".join(f"{REFERENCE[k][t]:>27}" for k in REFERENCE) + f" | {ours[t]:>23}")
print(f"{'ALL':10s} | {sum(counts.values()):>5} | " + " | ".join(f"{sum(REFERENCE[k].values()):>27}" for k in REFERENCE)
      + f" | {sum(ours.values()):>23}")
sc = [r for r in rows if r["type"] == "score"]
if sc:
    print(f"\nscore: exact level {sum(r['level_error'] == 0 for r in sc)} / {len(sc)} | within 1 level {sum(r['level_error'] <= 1 for r in sc)} / {len(sc)} | "
          f"mean |expected - true| {sum(abs(r['expected_level'] - r['answer_index']) for r in sc) / len(sc):.2f} levels")
yn = [r for r in rows if r["type"] == "noul"]
print(f"yes / no: answered yes {sum(r['pick'] == 'yes' for r in yn)} of {len(yn)} (truth: {sum(r['answer'] == 'yes' for r in yn)} yes)")

print("\nper category:")
for cat in sorted({r["category"] for r in rows}):
    rr = [r for r in rows if r["category"] == cat]
    print(f"  {cat:10s} {sum(r['ok'] for r in rr):>3} / {len(rr)}")

groups = {}
for r in rows:
    if "pair" in r:
        groups.setdefault(r["pair"], []).append(r)
print(f"\nquestion groups (same text, different questions): all answers right in {sum(all(r['ok'] for r in g) for g in groups.values())} of {len(groups)}")
for pid, g in groups.items():
    print(f"  {pid}: " + " | ".join(f"{r['question'][:34]} -> {r['pick']}{'' if r['ok'] else ' x'}" for r in g))

print("\nmistakes:")
for r in [r for r in rows if not r["ok"]]:
    s_ = r["state"] if len(r["state"]) <= 46 else r["state"][:43] + "..."
    print(f"  {r['id']} {r['type']:6s} | {s_:46s} | {r['question'][:38]:38s} | expected {r['answer']:14s} | got {r['pick']} ({max(r['probs']):.2f})")

with open("e12_lora.json", "w") as fh:
    json.dump({"encoder": ENCODER, "lora_r": LORA_R, "epochs": EPOCHS, "best_epoch": best[2], "ms_per_question": ms,
               "train_minutes": train_time / 60, "rows": [{k: r[k] for k in ("id", "type", "category", "pick", "ok", "probs")} for r in rows]}, fh)
torch.save({"lora": {n: p.detach().cpu() for n, p in encoder.named_parameters() if p.requires_grad},
            "head": head.state_dict(), "encoder": ENCODER}, "choice_lora_weights.pt")
log("saved e12_lora.json and choice_lora_weights.pt")

[08:20:58] E12: 170 questions, 49 ms per question

E12        | items |    run 14 frozen, zero-shot | run 17 frozen + choice head |          Strands decider 2B |  LoRA + head (this run)
noul       |    58 |                          29 |                          38 |                          56 |                      52
choice     |    59 |                          39 |                          44 |                          56 |                      55
score      |    53 |                          28 |                          32 |                          48 |                      46
ALL        |   170 |                          96 |                         114 |                         160 |                     153

score: exact level 46 / 53 | within 1 level 53 / 53 | mean |expected - true| 0.13 levels
yes / no: answered yes 30 of 58 (truth: 30 yes)

per category:
  compare     10 / 12
  detail      37 / 38
  fact        34 / 37
  intent      20 / 22
  language     2 / 2
  routing   